# 08 Results And Plot Interpretation

This notebook is the single downstream review notebook for the Dutch 15-minute DAM extension.

Use it when you want one place that:

- explains how the hourly anchor models, quarter-hour shape models, and counterfactual datasets were constructed;
- shows the final empirical forecast results under both oracle and realistic anchors;
- visualises representative observed 15-minute periods and counterfactual full-year periods;
- reports the frozen canonical actual-path registry that later bidding work must use.

The notebook is intentionally interpretation-heavy. It does not replace the phase notebooks. It condenses them into one reproducible readout.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Refresh Hooks

By default this notebook only reads the latest saved artifacts. Set any flag to `True` if you want to rerun the corresponding downstream phase before loading the summary views.

In [ ]:
RUN_PHASE07 = False

phase_scripts = [
    (RUN_PHASE07, "run_15min_phase07_realistic_track_a.py", "Phase 7"),
]

for should_run, script_name, label in phase_scripts:
    if not should_run:
        continue
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / script_name),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"{label} rerun failed with exit code {completed.returncode}.")

In [ ]:
VERSION_ID = "canonical_v1"

latest_phase03 = find_latest_phase03_run(config)
latest_phase04 = find_latest_phase04_run(config)
latest_phase07 = find_latest_phase07_run(config)
latest_upstream = find_latest_phase07_upstream_refresh_run(config)
canonical_version_dir = find_frozen_actual_version(config, VERSION_ID)

required_runs = {
    "phase03": latest_phase03,
    "phase04": latest_phase04,
    "phase07": latest_phase07,
    "canonical_actual": canonical_version_dir,
}
missing = [name for name, path in required_runs.items() if path is None]
if missing:
    raise FileNotFoundError(f"Missing required saved artifacts for: {missing}")

phase03_selected = pd.read_csv(latest_phase03 / "selected_anchor_models.csv")
phase03_summary = pd.read_csv(latest_phase03 / "candidate_selection_summary.csv")

phase04_price_metrics = pd.read_csv(latest_phase04 / "reconstructed_price_metrics.csv")
phase04_shape_metrics = pd.read_csv(latest_phase04 / "shape_only_metrics.csv")
phase04_model_config = pd.read_csv(latest_phase04 / "model_configuration_summary.csv")
phase04_tuning = pd.read_csv(latest_phase04 / "tuning_results.csv")
phase04_recommended = pd.read_csv(latest_phase04 / "recommended_model_summary.csv")

phase07_status = pd.read_csv(latest_phase07 / "status_summary.csv")
phase07_checks = pd.read_csv(latest_phase07 / "validation_checks.csv")
phase07_hourly_metrics = pd.read_csv(latest_phase07 / "hourly_anchor_metrics.csv")
phase07_hourly_coverage = pd.read_csv(latest_phase07 / "hourly_anchor_coverage_summary.csv")
phase07_price_metrics = pd.read_csv(latest_phase07 / "reconstructed_price_metrics.csv")
phase07_shape_metrics = pd.read_csv(latest_phase07 / "shape_only_metrics.csv")
phase07_mae_by_hour = pd.read_csv(latest_phase07 / "mae_by_hour.csv")
phase07_perf_by_condition = pd.read_csv(latest_phase07 / "performance_by_condition.csv")
phase07_predictions = pd.read_csv(latest_phase07 / "predictions_long.csv")
phase07_model_config = pd.read_csv(latest_phase07 / "model_configuration_summary.csv")
phase07_tuning = pd.read_csv(latest_phase07 / "tuning_results.csv")
phase07_comparison = pd.read_csv(latest_phase07 / "oracle_vs_realistic_comparison.csv")
phase07_recommended = pd.read_csv(latest_phase07 / "recommended_model_summary.csv")

upstream_freshness = pd.read_csv(latest_upstream / "freshness_summary.csv") if latest_upstream is not None else pd.DataFrame()
canonical_entry = resolve_frozen_actual_registry_entry(config, version_id=VERSION_ID, verify_hash=True)
canonical_manifest = load_frozen_actual_manifest(config, version_id=VERSION_ID, verify_hash=False)
canonical_provenance = build_thesis_grade_frozen_actual_metadata(config, version_id=VERSION_ID, verify_hash=False)
canonical_actual = load_frozen_actual_path(config, version_id=VERSION_ID, verify_hash=False)
canonical_diagnostics = load_frozen_actual_diagnostics(config, version_id=VERSION_ID, verify_hash=False)

run_paths = pd.DataFrame(
    [
        {"phase": "03", "artifact_path": str(latest_phase03)},
        {"phase": "04", "artifact_path": str(latest_phase04)},
        {"phase": "07", "artifact_path": str(latest_phase07)},
        {"phase": "canonical_actual", "artifact_path": str(canonical_version_dir)},
        {"phase": "07_upstream_refresh", "artifact_path": str(latest_upstream) if latest_upstream is not None else ""},
    ]
)
display(run_paths)

## What Is Being Summarised Here

The results come from three distinct layers. Keeping them separate matters:

1. **Observed 15-minute empirical validation**
   The model is tested against real Dutch quarter-hour prices from the post-implementation period.

2. **Realistic end-to-end forecasting**
   The hour-level anchor is no longer known. It must first be forecast by the selected hourly models, then refined by the quarter-hour shape model.

3. **Counterfactual full-year construction**
   The official hourly test year predates the Dutch quarter-hour market. Any 15-minute series for that period is synthetic or counterfactual by design.

In [ ]:
phase_status_rows = [
    {"phase": "03", "purpose": "Dynamic hourly anchor selection", "status": "completed", "run_path": str(latest_phase03)},
    {"phase": "04", "purpose": "Observed quarter-hour oracle validation", "status": "completed", "run_path": str(latest_phase04)},
    {"phase": "canonical_actual", "purpose": "Frozen canonical actual-path registry", "status": "completed", "run_path": str(canonical_version_dir)},
    {"phase": "07", "purpose": "Realistic end-to-end Track A validation", "status": str(phase07_status.iloc[0].get("phase07_status", "completed")), "run_path": str(latest_phase07)},
]
phase_status = pd.DataFrame(phase_status_rows)
display(phase_status)

dataset_regimes = pd.DataFrame(
    [
        {
            "regime": "Observed 15-minute empirical data",
            "period": "2025-10-01 to 2026-04-30",
            "used_for": "Shape-model training and empirical validation",
            "can_be_called_actual": "yes",
        },
        {
            "regime": "Realistic Track A end-to-end forecasts",
            "period": "2026-01-01 to 2026-04-30 validation/test windows",
            "used_for": "True observed quarter-hour forecast evaluation",
            "can_be_called_actual": "forecast compared against actual",
        },
        {
            "regime": "Frozen canonical 15-minute realized path",
            "period": "2024-10-01 to 2025-09-30",
            "used_for": "Full-year bidding comparisons in a synthetic quarter-hour market design",
            "can_be_called_actual": "no",
        },
    ]
)
display(dataset_regimes)

## Time Regimes At A Glance

The plot below is the quickest way to explain why the notebook contains both empirical forecast results and counterfactual scenario results.

In [ ]:
timeline = pd.DataFrame(
    [
        {"label": "Observed 15-min train", "start": "2025-10-01", "end": "2025-12-31", "band": "observed"},
        {"label": "Observed 15-min validation", "start": "2026-01-01", "end": "2026-02-28", "band": "observed"},
        {"label": "Observed 15-min test", "start": "2026-03-01", "end": "2026-04-30", "band": "observed"},
        {"label": "Official hourly test period", "start": "2024-10-01", "end": "2025-09-30", "band": "counterfactual"},
    ]
)
timeline["start"] = pd.to_datetime(timeline["start"])
timeline["end"] = pd.to_datetime(timeline["end"])
colors = {"observed": "#1f77b4", "counterfactual": "#ff7f0e"}

fig, ax = plt.subplots(figsize=(10.5, 3.8))
for idx, row in timeline.iloc[::-1].reset_index(drop=True).iterrows():
    ax.barh(
        y=row["label"],
        width=(row["end"] - row["start"]).days + 1,
        left=row["start"],
        color=colors[row["band"]],
        alpha=0.85,
    )
ax.set_title("Observed Validation Window vs Counterfactual Official Test Year")
ax.set_xlabel("Calendar date")
plt.tight_layout()
plt.show()

## How The Models Came About

The downstream 15-minute work did not start from scratch. It inherits the hourly model-selection logic and then adds a separate quarter-hour shape layer.

In [ ]:
lineage = pd.DataFrame(
    [
        {
            "layer": "Hourly anchor discovery",
            "artifact_source": "Phase 3",
            "how_selected": "Existing hourly scenario-selection logic reused without hardcoding candidates",
            "output": "Deterministic winner and hour-ranking winner",
        },
        {
            "layer": "Oracle shape validation",
            "artifact_source": "Phase 4",
            "how_selected": "Flat, mean-shape, LEAR, and XGBoost compared under the observed hourly mean",
            "output": "Pure shape-learning quality check",
        },
        {
            "layer": "Realistic end-to-end validation",
            "artifact_source": "Phase 7",
            "how_selected": "Selected hourly anchors combined with the same shape model ladder",
            "output": "True observed quarter-hour forecast performance",
        },
        {
            "layer": "Counterfactual full-year generator",
            "artifact_source": "Frozen canonical actual-path registry",
            "how_selected": "Pre-specified empirical block-sampling protocol frozen as canonical_v1 before downstream bidding",
            "output": "Authoritative thesis-grade synthetic 15-minute realized path",
        },
    ]
)
display(lineage)
display(phase03_selected)

## Training And Tuning Contract

The next tables show the actual train/validation/test windows and the modest tuning choices used for the shape layer. The point here is transparency, not hyperparameter theatre.

In [ ]:
display(phase04_model_config)
display(phase07_model_config)

oracle_best_tuning = (
    phase04_tuning.sort_values(["model", "validation_price_mae"])
    .groupby("model", as_index=False)
    .head(1)
)
realistic_best_tuning = (
    phase07_tuning.sort_values(["candidate_key", "model", "validation_price_mae"])
    .groupby(["candidate_key", "model"], as_index=False)
    .head(1)
)
display(oracle_best_tuning)
display(realistic_best_tuning)

## Main Result At A Glance

This is the realistic result that matters most for downstream interpretation. It includes both the hourly anchor error and the quarter-hour shape error.

In [ ]:
realistic_test = phase07_price_metrics[phase07_price_metrics["dataset_split"].astype(str) == "test"].copy()
realistic_test["anchor_model_pair"] = realistic_test["hourly_anchor_candidate_label"].astype(str) + " | " + realistic_test["model"].astype(str)
realistic_test = realistic_test.sort_values(["mae", "hourly_anchor_candidate_label", "model"]).reset_index(drop=True)
display(realistic_test[[
    "hourly_anchor_candidate_label",
    "hourly_anchor_role",
    "model",
    "mae",
    "rmse",
    "bias",
    "rmae_vs_flat_repeat",
    "cheapest_quarter_hit_rate",
    "most_expensive_quarter_hit_rate",
    "within_hour_rank_corr",
]])

best_realistic = realistic_test.iloc[0:1].copy()
display(best_realistic)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
bar_colors = ["#2ca02c" if model == "xgboost_shape" else "#7f7f7f" for model in realistic_test["model"]]
ax.bar(realistic_test["anchor_model_pair"], realistic_test["mae"], color=bar_colors)
ax.set_title("Realistic End-To-End Test MAE By Hourly Anchor And Shape Model")
ax.set_ylabel("MAE (EUR/MWh)")
ax.set_xlabel("Hourly anchor | shape model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Why Oracle And Realistic Results Differ

The oracle setup fixes the hourly level to the truth. The realistic setup does not. That is why the oracle MAE is much lower and should never be interpreted as the full forecasting error.

In [ ]:
comparison_plot = phase07_comparison[
    phase07_comparison["model"].isin(["flat_repeat", "mean_shape", "xgboost_shape"])
].copy()
comparison_plot["label"] = comparison_plot["hourly_anchor_candidate_label"].astype(str) + "\n" + comparison_plot["model"].astype(str)

x = np.arange(len(comparison_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(11.0, 4.8))
ax.bar(x - width / 2, comparison_plot["oracle_test_mae"], width=width, label="oracle anchor")
ax.bar(x + width / 2, comparison_plot["mae"], width=width, label="realistic anchor")
ax.set_xticks(x)
ax.set_xticklabels(comparison_plot["label"], rotation=25, ha="right")
ax.set_ylabel("MAE (EUR/MWh)")
ax.set_title("Oracle vs Realistic Test MAE")
ax.legend()
plt.tight_layout()
plt.show()

## How Much Of The Final Error Comes From The Hourly Anchor

This plot compares the hourly anchor MAE against the final best 15-minute MAE for each selected hourly candidate. It is a clean way to show that the shape layer helps, but the hourly level still dominates.

In [ ]:
hourly_test = phase07_hourly_metrics[phase07_hourly_metrics["dataset_split"].astype(str) == "test"].copy()
best_shape_per_anchor = (
    realistic_test.sort_values("mae")
    .groupby(["hourly_anchor_candidate_key", "hourly_anchor_candidate_label"], as_index=False)
    .first()[["hourly_anchor_candidate_key", "hourly_anchor_candidate_label", "model", "mae"]]
    .rename(columns={"model": "best_shape_model", "mae": "best_15min_test_mae"})
)
anchor_vs_final = hourly_test.merge(
    best_shape_per_anchor,
    left_on=["candidate_key", "candidate_label"],
    right_on=["hourly_anchor_candidate_key", "hourly_anchor_candidate_label"],
    how="left",
)
display(anchor_vs_final[["candidate_label", "mae", "best_shape_model", "best_15min_test_mae"]].rename(columns={"mae": "hourly_anchor_test_mae"}))

x = np.arange(len(anchor_vs_final))
width = 0.35
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.bar(x - width / 2, anchor_vs_final["mae"], width=width, label="hourly anchor MAE")
ax.bar(x + width / 2, anchor_vs_final["best_15min_test_mae"], width=width, label="best final 15-min MAE")
ax.set_xticks(x)
ax.set_xticklabels(anchor_vs_final["candidate_label"], rotation=15, ha="right")
ax.set_ylabel("MAE (EUR/MWh)")
ax.set_title("Hourly Anchor Error vs Final 15-Minute Error")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def _with_local_timestamps(frame: pd.DataFrame, column: str) -> pd.Series:
    return pd.to_datetime(frame[column], utc=True).dt.tz_convert(config.business_timezone)


def _iso_week_id(ts: pd.Series) -> pd.Series:
    iso = ts.dt.isocalendar()
    return iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2)


def _weekly_summary_from_prices(frame: pd.DataFrame, *, timestamp_col: str, price_col: str) -> pd.DataFrame:
    data = frame.copy()
    data["_ts"] = _with_local_timestamps(data, timestamp_col)
    data["iso_week_id"] = _iso_week_id(data["_ts"])
    data["week_start"] = (data["_ts"] - pd.to_timedelta(data["_ts"].dt.weekday, unit="D")).dt.normalize()
    summary = (
        data.groupby("iso_week_id", as_index=False)
        .agg(
            week_start=("week_start", "min"),
            mean_price=(price_col, "mean"),
            std_price=(price_col, "std"),
            min_price=(price_col, "min"),
            max_price=(price_col, "max"),
            negative_share=(price_col, lambda s: float((s < 0).mean())),
        )
    )
    summary["price_range"] = summary["max_price"] - summary["min_price"]
    return summary.sort_values("week_start").reset_index(drop=True)


def _choose_representative_week(summary: pd.DataFrame) -> str:
    feature_cols = ["mean_price", "std_price", "price_range", "negative_share"]
    scaled = summary[feature_cols].copy()
    scaled = (scaled - scaled.mean()) / scaled.std(ddof=0).replace(0, 1.0)
    distances = np.sqrt((scaled**2).sum(axis=1))
    return str(summary.loc[distances.idxmin(), "iso_week_id"])


def _choose_high_vol_week(summary: pd.DataFrame) -> str:
    return str(summary.sort_values(["std_price", "week_start"], ascending=[False, True]).iloc[0]["iso_week_id"])


def _choose_low_price_week(summary: pd.DataFrame) -> str:
    ordered = summary.sort_values(["negative_share", "mean_price", "week_start"], ascending=[False, True, True])
    return str(ordered.iloc[0]["iso_week_id"])


def _choose_seasonal_week(summary: pd.DataFrame, *, months: tuple[int, ...]) -> str:
    seasonal = summary[summary["week_start"].dt.month.isin(months)].copy()
    if seasonal.empty:
        return _choose_representative_week(summary)
    return _choose_representative_week(seasonal)


def _plot_observed_week(ax, week_data: pd.DataFrame, title: str) -> None:
    actual = week_data.drop_duplicates(subset=["timestamp_utc"])[["timestamp_local", "price_eur_per_mwh", "hourly_forecast_anchor_price_eur_per_mwh"]].sort_values("timestamp_local")
    actual_ts = _with_local_timestamps(actual, "timestamp_local")
    ax.plot(actual_ts, actual["price_eur_per_mwh"], label="actual 15-min", linewidth=2.0, color="black")
    ax.step(
        actual_ts,
        actual["hourly_forecast_anchor_price_eur_per_mwh"],
        where="post",
        label="hourly anchor",
        linestyle="--",
        color="#9467bd",
    )
    for model_name, color in [("flat_repeat", "#7f7f7f"), ("mean_shape", "#ff7f0e"), ("xgboost_shape", "#2ca02c")]:
        model_slice = (
            week_data[week_data["model"].astype(str) == model_name][["timestamp_local", "predicted_price_eur_per_mwh"]]
            .sort_values("timestamp_local")
        )
        if not model_slice.empty:
            ax.plot(_with_local_timestamps(model_slice, "timestamp_local"), model_slice["predicted_price_eur_per_mwh"], label=model_name, alpha=0.95, color=color)
    ax.set_title(title)
    ax.set_ylabel("EUR/MWh")


def _plot_counterfactual_week(ax, week_data: pd.DataFrame, title: str) -> None:
    actual_hourly = (
        week_data.drop_duplicates(subset=["hour_start_utc"])[["hour_start_local", "hourly_anchor_price_eur_per_mwh"]]
        .sort_values("hour_start_local")
    )
    ax.step(
        _with_local_timestamps(actual_hourly, "hour_start_local"),
        actual_hourly["hourly_anchor_price_eur_per_mwh"],
        where="post",
        label="actual hourly anchor",
        color="black",
        linewidth=2.0,
    )
    color_lookup = {
        "flat": "#7f7f7f",
        "empirical_medium": "#1f77b4",
        "high_volatility": "#ff7f0e",
        "stress": "#d62728",
        "canonical_v1": "#1f77b4",
    }
    variants = [str(value) for value in week_data["scenario_variant"].dropna().astype(str).drop_duplicates().tolist()]
    for idx, scenario_variant in enumerate(variants):
        color = color_lookup.get(scenario_variant, plt.cm.tab10(idx % 10))
        scenario_slice = (
            week_data[week_data["scenario_variant"].astype(str) == scenario_variant][["timestamp_local", "predicted_price_eur_per_mwh"]]
            .sort_values("timestamp_local")
        )
        if not scenario_slice.empty:
            ax.plot(_with_local_timestamps(scenario_slice, "timestamp_local"), scenario_slice["predicted_price_eur_per_mwh"], label=scenario_variant, color=color, alpha=0.9)
    ax.set_title(title)
    ax.set_ylabel("EUR/MWh")

## Observed 15-Minute Case Weeks

The next plots stay inside the observed test window. That keeps them aligned with the realistic results above.

In [ ]:
best_anchor_label = str(best_realistic.iloc[0]["hourly_anchor_candidate_label"])
best_model_name = str(best_realistic.iloc[0]["model"])
observed_best = phase07_predictions[
    (phase07_predictions["dataset_split"].astype(str) == "test")
    & (phase07_predictions["anchor_mode"].astype(str) == "realistic_forecast")
    & (phase07_predictions["hourly_anchor_candidate_label"].astype(str) == best_anchor_label)
].copy()
observed_best["timestamp_local_dt"] = _with_local_timestamps(observed_best, "timestamp_local")
observed_best["iso_week_id"] = _iso_week_id(observed_best["timestamp_local_dt"])

observed_actual = observed_best.drop_duplicates(subset=["timestamp_utc"])[["timestamp_local", "price_eur_per_mwh"]].copy()
observed_summary = _weekly_summary_from_prices(observed_actual, timestamp_col="timestamp_local", price_col="price_eur_per_mwh")
observed_case_weeks = pd.DataFrame(
    [
        {"case_type": "representative_observed_test_week", "iso_week_id": _choose_representative_week(observed_summary)},
        {"case_type": "high_volatility_observed_test_week", "iso_week_id": _choose_high_vol_week(observed_summary)},
        {"case_type": "low_price_observed_test_week", "iso_week_id": _choose_low_price_week(observed_summary)},
    ]
)
display(observed_summary)
display(observed_case_weeks)

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12.0, 10.5), sharex=False)
for ax, (_, row) in zip(axes, observed_case_weeks.iterrows()):
    week_id = str(row["iso_week_id"])
    week_slice = observed_best[observed_best["iso_week_id"].astype(str) == week_id].copy()
    _plot_observed_week(ax, week_slice, f"{row['case_type']} ({week_id})")
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, ncol=5, loc="upper right")
axes[-1].set_xlabel("Local timestamp")
plt.tight_layout()
plt.show()

## Selected Day Decomposition

This day-level view shows what the shape layer adds on top of the hourly anchor. The upper panel shows prices. The lower panel shows the predicted and actual within-hour deviations.

In [ ]:
xgb_test = observed_best[observed_best["model"].astype(str) == best_model_name].copy()
daily_spread = (
    xgb_test.groupby("hour_local_date", as_index=False)
    .agg(
        actual_day_min=("price_eur_per_mwh", "min"),
        actual_day_max=("price_eur_per_mwh", "max"),
    )
)
daily_spread["actual_day_spread"] = daily_spread["actual_day_max"] - daily_spread["actual_day_min"]
selected_day = str(daily_spread.sort_values(["actual_day_spread", "hour_local_date"], ascending=[False, True]).iloc[0]["hour_local_date"])
day_slice = observed_best[observed_best["hour_local_date"].astype(str) == selected_day].copy()
actual_day = day_slice.drop_duplicates(subset=["timestamp_utc"])[["timestamp_local", "price_eur_per_mwh", "hourly_forecast_anchor_price_eur_per_mwh", "delta_eur_per_mwh"]].sort_values("timestamp_local")
model_day = day_slice[day_slice["model"].astype(str) == best_model_name][["timestamp_local", "predicted_price_eur_per_mwh", "delta_pred_adjusted"]].sort_values("timestamp_local")
actual_day_ts = _with_local_timestamps(actual_day, "timestamp_local")
model_day_ts = _with_local_timestamps(model_day, "timestamp_local")

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(11.5, 7.0), sharex=True, gridspec_kw={"height_ratios": [2.0, 1.0]})
axes[0].plot(actual_day_ts, actual_day["price_eur_per_mwh"], label="actual 15-min", color="black", linewidth=2.0)
axes[0].step(actual_day_ts, actual_day["hourly_forecast_anchor_price_eur_per_mwh"], where="post", label="hourly anchor", linestyle="--", color="#9467bd")
axes[0].plot(model_day_ts, model_day["predicted_price_eur_per_mwh"], label=f"{best_model_name}", color="#2ca02c")
axes[0].set_title(f"Observed Day Decomposition: {selected_day}")
axes[0].set_ylabel("Price (EUR/MWh)")
axes[0].legend()

axes[1].plot(actual_day_ts, actual_day["delta_eur_per_mwh"], label="actual delta", color="black", linewidth=1.8)
axes[1].plot(model_day_ts, model_day["delta_pred_adjusted"], label="predicted delta", color="#2ca02c")
axes[1].axhline(0.0, color="grey", linewidth=1.0)
axes[1].set_ylabel("Delta (EUR/MWh)")
axes[1].set_xlabel("Local timestamp")
axes[1].legend()
plt.tight_layout()
plt.show()

## Decision-Relevant Quarter-Hour Interpretation

The next pair of plots focuses on what the MILP actually cares about: when the model gets the cheapest quarter right, and at which hours of day the error remains stubborn.

In [ ]:
best_rows = observed_best[observed_best["model"].astype(str) == best_model_name].copy()
rank_group = (
    best_rows.groupby("hour_start_utc", as_index=False)
    .agg(
        actual_cheapest_quarter=("price_eur_per_mwh", lambda s: int(np.argmin(s.to_numpy()) + 1)),
        predicted_cheapest_quarter=("predicted_price_eur_per_mwh", lambda s: int(np.argmin(s.to_numpy()) + 1)),
    )
)
cheapest_confusion = pd.crosstab(
    rank_group["actual_cheapest_quarter"],
    rank_group["predicted_cheapest_quarter"],
    normalize="index",
).reindex(index=[1, 2, 3, 4], columns=[1, 2, 3, 4], fill_value=0.0)

mae_hour_plot = phase07_mae_by_hour[
    (phase07_mae_by_hour["hourly_anchor_candidate_label"].astype(str) == best_anchor_label)
    & (phase07_mae_by_hour["model"].astype(str).isin(["flat_repeat", "mean_shape", best_model_name]))
].copy()

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12.0, 4.4))
im = axes[0].imshow(cheapest_confusion.values, cmap="Blues", vmin=0.0, vmax=1.0)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([1, 2, 3, 4])
axes[0].set_yticks(range(4))
axes[0].set_yticklabels([1, 2, 3, 4])
axes[0].set_xlabel("Predicted cheapest quarter")
axes[0].set_ylabel("Actual cheapest quarter")
axes[0].set_title("Cheapest-Quarter Hit Matrix")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

for model_name, group in mae_hour_plot.groupby("model"):
    axes[1].plot(group["local_hour_of_day"], group["mae"], marker="o", label=model_name)
axes[1].set_title("Test MAE By Hour Of Day")
axes[1].set_xlabel("Local hour")
axes[1].set_ylabel("MAE (EUR/MWh)")
axes[1].legend()
plt.tight_layout()
plt.show()

## Counterfactual Official-Year Views

These plots are not forecast-validation plots. They are interpretation plots for the synthetic 15-minute market-design dataset built on the official hourly test year.

In [ ]:
official_hourly = canonical_actual[["hour_start_local", "observed_hourly_anchor_price_eur_per_mwh"]].drop_duplicates(subset=["hour_start_local"]).copy()
official_hourly = official_hourly.rename(
    columns={
        "hour_start_local": "timestamp_local",
        "observed_hourly_anchor_price_eur_per_mwh": "price_eur_per_mwh",
    }
)
official_hourly["timestamp_local_dt"] = _with_local_timestamps(official_hourly, "timestamp_local")
official_weekly = _weekly_summary_from_prices(official_hourly, timestamp_col="timestamp_local", price_col="price_eur_per_mwh")
official_case_weeks = pd.DataFrame(
    [
        {"case_type": "typical_winter_official_year", "iso_week_id": _choose_seasonal_week(official_weekly, months=(12, 1, 2))},
        {"case_type": "typical_summer_official_year", "iso_week_id": _choose_seasonal_week(official_weekly, months=(6, 7, 8))},
        {"case_type": "high_volatility_official_year", "iso_week_id": _choose_high_vol_week(official_weekly)},
    ]
)
display(official_weekly)
display(official_case_weeks)

counterfactual_plot = canonical_actual.copy()
counterfactual_plot["hourly_anchor_price_eur_per_mwh"] = pd.to_numeric(counterfactual_plot["observed_hourly_anchor_price_eur_per_mwh"], errors="coerce")
counterfactual_plot["predicted_price_eur_per_mwh"] = pd.to_numeric(counterfactual_plot["counterfactual_actual_price_eur_per_mwh"], errors="coerce")
counterfactual_plot["scenario_variant"] = VERSION_ID
counterfactual_plot["timestamp_local_dt"] = _with_local_timestamps(counterfactual_plot, "timestamp_local")
counterfactual_plot["iso_week_id"] = _iso_week_id(counterfactual_plot["timestamp_local_dt"])

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12.0, 10.5), sharex=False)
for ax, (_, row) in zip(axes, official_case_weeks.iterrows()):
    week_id = str(row["iso_week_id"])
    week_slice = counterfactual_plot[counterfactual_plot["iso_week_id"].astype(str) == week_id].copy()
    _plot_counterfactual_week(ax, week_slice, f"{row['case_type']} ({week_id})")
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, ncol=5, loc="upper right")
axes[-1].set_xlabel("Local timestamp")
plt.tight_layout()
plt.show()

## Frozen Canonical Actual-Path Registry

The tables below show the authoritative canonical actual-path metadata that later bidding work must report.

In [ ]:
display(pd.DataFrame([canonical_provenance]))
manifest_rows = []
for key, value in canonical_manifest.items():
    manifest_rows.append(
        {
            "field": key,
            "value": json.dumps(value, default=str) if isinstance(value, (dict, list)) else value,
        }
    )
display(pd.DataFrame(manifest_rows))
display(canonical_diagnostics)

## Legacy Exploratory Path Status

Legacy Phase 5/6 outputs remain useful for exploratory diagnostics, but they are not authorised actual market truth for thesis-grade downstream runs.

In [ ]:
legacy_status = pd.DataFrame(
    [
        {
            "legacy_component": "Phase 5 counterfactual generation",
            "status": "exploratory_only",
            "authorised_as_15min_actual_market_truth": False,
        },
        {
            "legacy_component": "Phase 6 mixed export bundle",
            "status": "exploratory_only",
            "authorised_as_15min_actual_market_truth": False,
        },
        {
            "legacy_component": "Frozen canonical actual-path registry",
            "status": "authoritative_thesis_grade",
            "authorised_as_15min_actual_market_truth": True,
        },
    ]
)
display(legacy_status)

## Final Interpretation

The main conclusions supported by the current results are:

- The downstream 15-minute extension now works end to end.
- The best learned quarter-hour shape layer is XGBoost.
- Mean-shape is a serious simple benchmark and should remain visible in the thesis.
- LEAR adds little as a quarter-hour shape learner in this setup.
- The realistic end-to-end error is dominated by the hourly anchor, not by the shape layer.
- The frozen canonical actual-path registry now provides the single authorised 15-minute realised market environment for thesis-grade downstream bidding work.

Claims that remain out of scope:

- claiming actual full-year 15-minute forecast accuracy for the official hourly test year;
- treating pre-October-2025 quarter-hour prices as historical truth;
- treating the short observed post-implementation period as proof of full-year seasonal quarter-hour behaviour.